# When More Replicas Do Not Fix Tail Latency

**Guiding question:** Riverside added model-server replicas. Why can p95 latency remain unchanged?

You already have a gateway, an inference server, and deployment machinery. This chapter asks whether their boundaries compose into one usable service.

## Before taking the Ch9 shortcut

You are skipping a replay of Chapters 01–08, not their capability requirements. Show that you can separate prompt processing (**prefill**) from token-by-token generation (**decode**), explain retained KV-cache memory, distinguish time to first token (**TTFT**) from time per later token (**TPOT**), and point to the gateway owner for a bounded deadline and retry policy. Use [inference systems](../07-inference-systems/inference-systems.ipynb) and the [gateway route](../../09-llm-gateway/README.md) if those checks fail.

Bring a workload record with environment, population, method, and limits; a labeled fixture is acceptable for learning mechanics, not for proving production capacity. At exit, retain the queue diagnosis, admission decision, scaling signal, and compatible rollback identity before [operational serving](../10-azure-operational-llm-serving/README.md).

## 0. Scope and evidence boundary

> **Evidence:** `LOCAL_FIXTURE` — mechanics demonstrated; production capability unavailable; live behavior unvalidated.

**Prerequisites:** [LLM Gateway](../../09-llm-gateway/README.md) and [Inference Systems](../07-inference-systems/inference-systems.ipynb).

| Pass | Question |
|---|---|
| Expose the failure | Does replica count help when an earlier gate still serializes work? |
| Align the path | What changes when gateway concurrency can feed every replica? |
| Bound overload | Can admission protect accepted requests without pretending capacity increased? |
| Assign ownership | Which layer owns routing, batching, memory, scaling, SLOs, and rollback? |

## 1. One request crosses several owners

```mermaid
flowchart LR
    C[Client] --> G[Gateway policy]
    G --> A[Admission queue]
    A --> S[Model scheduler]
    S --> R[Replica pool]
    R --> T[Tokens and trace]
    M[Metrics and SLOs] --> P[Autoscaler]
    P --> R
    T --> M
```

A replica is only one link. If the gateway admits one in-flight request, three replicas still receive one request at a time.

## 2. Attempt 1: add replicas and hope

**Predict:** Riverside changes from one replica to three, but the gateway's maximum in-flight work remains one. Which result should change: throughput, p95 latency, both, or neither?

The fixture below creates four deterministic request bursts. Service time depends on requested output tokens. It is a queue model, not a real inference benchmark.

In [ ]:
from heapq import heapify, heappop, heappush


def percentile(values, q):
    ordered = sorted(values)
    position = (len(ordered) - 1) * q
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] * (1 - fraction) + ordered[upper] * fraction


def make_bursty_requests(requests_per_wave=12):
    requests = []
    for wave in range(4):
        base_time = wave * 1.2
        for index in range(requests_per_wave):
            requests.append({
                "id": f"wave-{wave}-request-{index}",
                "arrival": base_time + index * 0.025,
                "tokens": 24 + (index % 4) * 8,
            })
    return requests


def simulate(requests, replicas, gateway_limit, max_queue_wait=None):
    effective_capacity = min(replicas, gateway_limit)
    available_at = [0.0] * effective_capacity
    heapify(available_at)
    rows, busy_time = [], 0.0

    for request in requests:
        next_available = heappop(available_at)
        queue_wait = max(0.0, next_available - request["arrival"])
        service_time = 0.12 + 0.004 * request["tokens"]

        if max_queue_wait is not None and queue_wait > max_queue_wait:
            heappush(available_at, next_available)
            rows.append({**request, "accepted": False, "queue_wait": queue_wait})
            continue

        start = max(request["arrival"], next_available)
        finish = start + service_time
        heappush(available_at, finish)
        busy_time += service_time
        rows.append({
            **request,
            "accepted": True,
            "queue_wait": queue_wait,
            "latency": finish - request["arrival"],
            "finish": finish,
        })

    accepted = [row for row in rows if row["accepted"]]
    latencies = [row["latency"] for row in accepted]
    waits = [row["queue_wait"] for row in accepted]
    span = max(row["finish"] for row in accepted) - requests[0]["arrival"]
    return {
        "requests": len(rows),
        "accepted": len(accepted),
        "rejected": len(rows) - len(accepted),
        "throughput": len(accepted) / span,
        "p50": percentile(latencies, 0.50),
        "p95": percentile(latencies, 0.95),
        "max_wait": max(waits),
        "utilization": busy_time / (span * effective_capacity),
    }


The simulator changes only three controls:

1. physical replica count;
2. the upstream in-flight limit;
3. an optional maximum queue wait.

It deliberately leaves batching, network time, failures, and token streaming out. That keeps the complaint inspectable.

In [ ]:
requests = make_bursty_requests()
scenarios = {
    "A: one replica": simulate(requests, replicas=1, gateway_limit=1),
    "B: three replicas, gate=1": simulate(requests, replicas=3, gateway_limit=1),
    "C: three replicas, gate=3": simulate(requests, replicas=3, gateway_limit=3),
    "D: gate=3 + admission": simulate(
        requests, replicas=3, gateway_limit=3, max_queue_wait=0.18
    ),
}

header = (
    f"{'scenario':<29} {'ok':>3} {'rej':>4} {'req/s':>7} "
    f"{'p50 s':>7} {'p95 s':>7} {'max q s':>8} {'util':>7}"
)
print(header)
print("-" * len(header))
for name, result in scenarios.items():
    print(
        f"{name:<29} {result['accepted']:>3} {result['rejected']:>4} "
        f"{result['throughput']:>7.2f} {result['p50']:>7.3f} "
        f"{result['p95']:>7.3f} {result['max_wait']:>8.3f} "
        f"{result['utilization']:>7.1%}"
    )

one = scenarios["A: one replica"]
constrained = scenarios["B: three replicas, gate=1"]
aligned = scenarios["C: three replicas, gate=3"]
admitted = scenarios["D: gate=3 + admission"]

assert one["p95"] == constrained["p95"]
assert one["throughput"] == constrained["throughput"]
assert aligned["p95"] < constrained["p95"]
assert admitted["max_wait"] <= 0.18
assert admitted["rejected"] > 0

print("\nTakeaway: replicas B existed, but the upstream gate made them unusable.")
print("Admission D bounded accepted queue wait by rejecting excess work explicitly.")


### That fixed capacity, but overload did not disappear

Scenario C feeds all three replicas and improves p95. It still accepts every request, even when the queue exceeds the service envelope.

Scenario D does **not** create capacity. It makes overload explicit: accepted requests stay inside the queue-wait budget, while excess work becomes a countable rejection the gateway can route, retry within policy, or surface.

## 3. Name the owner before tuning the mechanism

| Layer | Owns | Must expose | Does not own |
|---|---|---|---|
| Gateway | Authentication context, eligibility, budgets, route selection, bounded retry/fallback, stable errors | End-to-end deadline, route, retry count, rejection reason | Decode-step batching or KV placement |
| Model server | Token scheduling, continuous batching, KV memory, decoding, model-level admission | Queue wait, TTFT, TPOT, active sequences, cache pressure, token throughput | Customer authority or cross-provider policy |
| Platform/autoscaler | Replica/process topology, readiness, capacity changes | Ready replicas, queue pressure, accelerator memory, scaling events | Request semantics or release approval |
| Release control | Immutable artifact set, cohort gates, rollback target | Model/tokenizer/config/runtime identity and release-scoped SLOs | Retrying an already committed side effect |

**Checkpoint:** Where does batching belong? Near execution, where active sequences and KV pressure are visible. Where does business eligibility belong? At the gateway, before expensive work is admitted.

## 4. A rollback is more than an older weights file

Retain one compatible deployment identity:

- checkpoint and architecture configuration;
- tokenizer and special-token IDs;
- quantization format and serving runtime;
- decoding defaults and safety policy versions;
- replica, scheduler, admission, and autoscaling configuration;
- evaluator and SLO policy versions;
- rollback target, idempotency records, and in-flight reconciliation rule.

A deployment rollback cannot undo an external action that already committed. That boundary returns in Stage 12 role practice.

## 5. Toy to real

| Teaching mechanism | Production counterpart | What changes | What does not change |
|---|---|---|---|
| Heap of available workers | Model-server scheduler and replica pool | Prefill/decode phases, batching, streams, failures | Work waits for bounded execution capacity |
| Gateway in-flight limit | Admission semaphore, queue, or policy tier | Tenant policy, deadlines, priority | An upstream gate can withhold downstream capacity |
| Queue-wait budget | Backpressure and overload response | Error contract, retry-after, fallback | Rejection must remain visible in SLO evidence |
| Fixture p95 | Release-scoped latency telemetry | Real workloads, warm-up, tokens, hardware | Tail evidence must name population and configuration |
| Manual replica count | Autoscaler | Delayed signals, cooldowns, quotas | Scaling needs a signal tied to the actual complaint |

This notebook establishes mechanics with a local fixture. A workload-specific server run is needed for local capability evidence; an authorized deployment is needed for live-cloud evidence.

## 6. Your turn: change demand, not the conclusion

**Predict:** Increase requests per wave from 12 to 16. Will aligned replicas still improve p95? Will the same 0.18-second admission budget reject more or fewer requests?

Run the next cell, compare it with the original table, then write two sentences:

1. diagnose which metric changed and why;
2. explain whether the result supports a queue-mechanics claim, a production-capacity claim, both, or neither.

In [ ]:
larger_burst = make_bursty_requests(requests_per_wave=16)
transfer_aligned = simulate(larger_burst, replicas=3, gateway_limit=3)
transfer_admitted = simulate(
    larger_burst, replicas=3, gateway_limit=3, max_queue_wait=0.18
)

for name, result in {
    "aligned": transfer_aligned,
    "aligned + admission": transfer_admitted,
}.items():
    print(
        f"{name:<20} accepted={result['accepted']:>2} "
        f"rejected={result['rejected']:>2} p95={result['p95']:.3f}s "
        f"max_queue_wait={result['max_wait']:.3f}s"
    )


## 7. Evidence scorecard and boundary

| Opening question | Fixture evidence | Status |
|---|---|---|
| Can replicas help behind a serializing gate? | Scenarios A and B have identical throughput and p95 | No |
| What unlocks the replicas? | Scenario C aligns upstream concurrency with replica capacity | Demonstrated locally |
| Can admission manufacture throughput? | Scenario D rejects excess work rather than hiding it in latency | No |
| What should drive scaling? | Queue, sequence, token, memory, and release-scoped signals tied to the complaint | Explained, not live-validated |

### Coverage ledger

- **Built and measured:** deterministic queueing, constrained capacity, aligned capacity, queue-wait admission, bounded transfer intervention.
- **Explained or illustrated:** ownership, batching boundary, scaling signals, deployment and rollback state.
- **Named and out of scope:** real continuous batching, KV allocation, network effects, cloud autoscaling, quota, regional capacity, and production SLOs.

### Practical failure modes

- scaling on CPU while queue delay grows;
- retrying overload until it becomes more overload;
- hiding rejected requests from latency evidence;
- letting gateway and server both retry or batch the same work;
- rolling back weights without tokenizer, runtime, or policy compatibility.

You now have the missing composition map. Continue to [Azure Operational LLM Serving](../10-azure-operational-llm-serving/README.md), where readiness, deadlines, idempotency, telemetry, and blue/green release gates exercise these boundaries through an Azure-shaped local service.